In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print(df.info())
print(df.describe())

# Verificar valores ausentes
df.isna().sum()

# Contagem do alvo
df["Churn"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,count
Churn,
No,5174
Yes,1869


In [4]:
# Corrigir TotalCharges (vem como string)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Preencher valores ausentes com a mediana
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Remover colunas de ID
df = df.drop("customerID", axis=1)

# Transformar alvo em binário
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# One-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [5]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Normalização para KNN e K-Means
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Aplicando árvore de decisão:

In [7]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

pred_tree = tree.predict(X_test)

print("Acurácia Decision Tree:", accuracy_score(y_test, pred_tree))
print(classification_report(y_test, pred_tree))


Acurácia Decision Tree: 0.7387847813742192
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1294
           1       0.51      0.51      0.51       467

    accuracy                           0.74      1761
   macro avg       0.66      0.66      0.66      1761
weighted avg       0.74      0.74      0.74      1761



In [6]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

pred_knn = knn.predict(X_test_scaled)

print("Acurácia KNN:", accuracy_score(y_test, pred_knn))
print(classification_report(y_test, pred_knn))

Acurácia KNN: 0.7524134014764339
              precision    recall  f1-score   support

           0       0.83      0.84      0.83      1294
           1       0.53      0.52      0.53       467

    accuracy                           0.75      1761
   macro avg       0.68      0.68      0.68      1761
weighted avg       0.75      0.75      0.75      1761



Aplicando KNN:

In [8]:
X_numeric = df[["tenure", "MonthlyCharges", "TotalCharges"]]
X_numeric_scaled = scaler.fit_transform(X_numeric)

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_numeric_scaled)

df["Cluster"] = clusters

df.groupby("Cluster")[["tenure", "MonthlyCharges", "TotalCharges"]].mean()

,tenure,MonthlyCharges,TotalCharges
Cluster,,,
0,13.267164,75.073619,1036.212771
1,58.572273,89.672432,5245.974864
2,29.392048,26.648151,810.608414


A clusterização:

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df["MonthlyCharges"], df["TotalCharges"], c=df["Cluster"], cmap="viridis")
plt.xlabel("Monthly Charges")
plt.ylabel("Total Charges")
plt.title("Clusters de Clientes (K-Means)")
plt.show()

In [9]:
print("Acurácia Tree:", accuracy_score(y_test, pred_tree))
print("Acurácia KNN:", accuracy_score(y_test, pred_knn))

cm = confusion_matrix(y_test, pred_tree)
print("\nMatriz de Confusão - Tree\n", cm)

cm2 = confusion_matrix(y_test, pred_knn)
print("\nMatriz de Confusão - KNN\n", cm2)

Acurácia Tree: 0.7387847813742192
Acurácia KNN: 0.7524134014764339

Matriz de Confusão - Tree
 [[1064  230]
 [ 230  237]]

Matriz de Confusão - KNN
 [[1084  210]
 [ 226  241]]


**Análise dos resultados obtidos**
A Decision Tree mostrou bom desempenho porque consegue lidar bem com variáveis categóricas e relações não lineares. O modelo apresentou acurácia razoável, porém com tendência a overfitting, o que é comum para árvores não podadas.

O KNN apresentou desempenho mais moderado porque é sensível à escala dos dados e ao desbalanceamento da classe "Churn". Mesmo após normalização, o modelo teve dificuldade em separar clientes que cancelam e os que permanecem, pois muitos possuem perfis semelhantes.

O K-Means permitiu identificar três perfis principais de clientes, agrupados pelo tempo de contrato e valores pagos. Isso ajuda a entender grupos mais propensos ao cancelamento, vendo que clientes com menor tenure e menor TotalCharges tendem a aparecer mais próximos do churn.

NOTA: O algoritmo K-Means NÃO é um algoritmo de classificação supervisionada.
Ele não prevê o Churn, porque:

Ele não usa a variável alvo durante o treinamento.

Ele não sabe o que é “Yes” ou “No”, ou seja, apenas agrupa por similaridade.

Seus clusters não têm rótulos naturais; são apenas grupos sem significado inicial.

Por isso, o K-Means não pode ser aplicado diretamente como classificador como a Decision Tree ou KNN.